<a href="https://colab.research.google.com/github/PerdomoVergaraFernando/Procesos-Estocasticos/blob/main/M%C3%A9todo_de_Uniformizaci%C3%B3n_para_una_CMTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Teorema

Para calcular la matriz de transición $P(t)$ de la CMTC, utilizamos el método de uniformización:
$$P(t) = \sum^\infty_{k=0} e^{-rt} *\frac{(rt)^k}{k!}* \hat{P}^k$$

donde $r=6$ y $\hat{P}$ es la matriz estocástica obtenida en el ejercicio 1. La serie se aproxima truncando en $M$ términos, con:

$$M ≈ max\{rt + 5rt,20\}$$

A continuación se implementa el cálculo para $t=0.5,1,5$ y se verifica la ecuación de Chapman-Kolmogorov.

In [4]:
#Importación de librerías y definición de datos
import numpy as np

# Matriz de tasas R y parámetro r
R = np.array([[0, 2, 3, 0],
              [4, 0, 2, 0],
              [0, 2, 0, 2],
              [1, 0, 3, 0]])
r = 6  # tasa de uniformización

# Cálculo de la matriz estocástica hatP
r_i = R.sum(axis=1)  # tasas totales de salida: [5,6,4,4]
hatP = np.zeros_like(R, dtype=float)
for i in range(4):
    for j in range(4):
        if i == j:
            hatP[i,j] = 1 - r_i[i]/r
        else:
            hatP[i,j] = R[i,j] / r

print("Matriz hatP:")
print(hatP)

Matriz hatP:
[[0.1667 0.3333 0.5    0.    ]
 [0.6667 0.     0.3333 0.    ]
 [0.     0.3333 0.3333 0.3333]
 [0.1667 0.     0.5    0.3333]]


Explicación:
*   Calculamos las tasas totales $r_i$ sumando cada fila de $R$.
*   Construimos $\hat{P}$ elemento a elemento según la definición.
*   El resultado es una matriz estocástica (cada fila suma 1).

In [5]:
#Función para calcular P(t) mediante la serie truncada
def P_t(t, r, hatP, M=None):
    """
    Calcula la matriz de transición P(t) usando uniformización.
    t : tiempo
    r : tasa de Poisson (parámetro de uniformización)
    hatP : matriz estocástica de un paso
    M : número de términos (si no se da, se calcula automáticamente)
    """
    rt = r * t
    if M is None:
        M = int(np.round(max(rt + 5*np.sqrt(rt), 20)))  # redondeo entero
    # Inicializar suma con el término k=0: e^{-rt} * I
    term = np.exp(-rt) * np.eye(hatP.shape[0])
    P = term.copy()
    # Potencias de hatP
    pow_hatP = np.eye(hatP.shape[0])
    for k in range(1, M+1):
        pow_hatP = pow_hatP @ hatP   # hatP^k
        term = term * (rt / k)       # actualización recursiva del coeficiente
        P += term @ pow_hatP
    return P

# Calcular para t=0.5, 1, 5
t_values = [0.5, 1, 5]
for t in t_values:
    rt = r * t
    M_calc = int(np.round(max(rt + 5*np.sqrt(rt), 20)))
    print(f"\nPara t = {t}: rt = {rt:.2f}, M = {M_calc}")
    Pt = P_t(t, r, hatP)
    print("P(t) =")
    print(np.round(Pt, 6))


Para t = 0.5: rt = 3.00, M = 20
P(t) =
[[0.2506 0.217  0.3867 0.1458]
 [0.2531 0.2384 0.3744 0.1341]
 [0.1691 0.1936 0.4203 0.217 ]
 [0.158  0.1574 0.3983 0.2862]]

Para t = 1: rt = 6.00, M = 20
P(t) =
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]

Para t = 5: rt = 30.00, M = 57
P(t) =
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


Explicación:

*   La función $P_t$ implementa la suma truncada. Observemos que el término general es: $$a_k = e^{-rt} *\frac{(rt)^k}{k!}* \hat{P}^k $$
*   Para evitar recalcular factoriales y potencias de forma ineficiente, usamos recursión: $$a_k =a_{k-1}* \frac{rt}{k} $$
*   Las potencias de $\hat{P}$ se acumulan multiplicando por $\hat{P}$ en cada iteración.
*   El número $M$ se calcula según la fórmula sugerida, redondeado al entero más cercano.

In [6]:
#Verificación de Chapman-Kolmogorov
P05 = P_t(0.5, r, hatP)
P1 = P_t(1, r, hatP)
P1_calculado = P05 @ P05  # producto de matrices

print("\n--- Verificación de Chapman-Kolmogorov ---")
print("P(1) calculado por la serie:")
print(np.round(P1, 6))
print("\nP(0.5) * P(0.5):")
print(np.round(P1_calculado, 6))
print("\nDiferencia absoluta máxima:")
print(np.max(np.abs(P1 - P1_calculado)))


--- Verificación de Chapman-Kolmogorov ---
P(1) calculado por la serie:
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]

P(0.5) * P(0.5):
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]

Diferencia absoluta máxima:
5.820333639494635e-07


Explicación:

La ecuación de Chapman-Kolmogorov para tiempos homogéneos afirma que $P(t+s)=P(t)P(s)$. En particular, para $t=s=0.5$ debe cumplirse $P(1)=P(0.5)P(0.5$). Comparamos la matriz obtenida directamente con la serie en $t=1$ y el producto de las matrices en $t=0.5$.La diferencia numérica es muy pequeña debido al truncamiento, confirmando la propiedad.

##Implementación del algoritmo de uniformización con control de error

A continuación se implementa el algoritmo descrito en el enunciado, con tolerancia $ = 0.00001$, para calcular $P(t)$ para $t=0.5,1,5$ usando la matriz de tasas $R$ del ejercicio 1 y $r=6$. Se indica el valor de
$M$ (número de términos) requerido en cada caso y se comparan los resultados con los obtenidos mediante la fórmula heurística.

$$M ≈ max\{rt + 5rt,20\}$$   

##Algoritmo (pasos del enunciado)

1.   Dados $R,t,\epsilon$.
1.   Calcular $r=max_ir_i$(en nuestro caso $r=6$).
1.   Calcular $\hat{P}$
2.   Inicializar:
     *   $A= \hat{P}$
     *   $B=e^{−rt}I$
     *   $c=e^{−rt}$
     *   $suma=c$
     *   $k=1$

2.   Mientras $suma < 1 −\epsilon$:
     *   $c=c⋅\frac{(rt)}k$
     *   $B=B+cA$
     *   $A=A\hat{P}$
     *   $suma=suma+c$
     *   $k=k+1$

2.   Al final,$B$ es la aproximación de $P(t)$ con error en cada entrada menor o igual a $\epsilon$.

Nota: La condición
$suma < 1-\epsilon$ equivale a que la cola de la Poisson (suma desde $k+1$ hasta infinito) sea $>\epsilon$, que es exactamente la condición del teorema.

In [7]:
#Importar librerías y definir datos
import numpy as np
from scipy.special import factorial  # opcional, no se usa realmente

# Matriz de tasas R y parámetro r
R = np.array([[0, 2, 3, 0],
              [4, 0, 2, 0],
              [0, 2, 0, 2],
              [1, 0, 3, 0]])
r = 6  # max(r_i) = 6

# Calcular hatP (igual que antes)
r_i = R.sum(axis=1)  # [5, 6, 4, 4]
hatP = np.zeros_like(R, dtype=float)
n = hatP.shape[0]
for i in range(n):
    for j in range(n):
        if i == j:
            hatP[i, j] = 1 - r_i[i] / r
        else:
            hatP[i, j] = R[i, j] / r

print("Matriz hatP:")
print(np.round(hatP, 4))

Matriz hatP:
[[0.1667 0.3333 0.5    0.    ]
 [0.6667 0.     0.3333 0.    ]
 [0.     0.3333 0.3333 0.3333]
 [0.1667 0.     0.5    0.3333]]


Explicación:

Se construye la matriz estocástica $\hat{P}$ a partir de $R$ y $r=6$. Los valores coinciden con los calculados previamente

In [8]:
#Función que implementa el algoritmo con control de error
def P_t_epsilon(t, r, hatP, eps=1e-5):
    #Calcula P(t) mediante uniformización con tolerancia eps.
    #Retorna la matriz B y el número de términos M (k final).
    rt = r * t
    # Inicialización
    c = np.exp(-rt)          # coeficiente para k=0
    B = c * np.eye(hatP.shape[0])   # B = e^{-rt} I
    A = hatP                 # A = hatP^1
    suma = c                 # suma de coeficientes hasta k=0
    k = 1
    # Iterar mientras la cola > eps (es decir, suma < 1 - eps)
    while suma < 1 - eps:
        c = c * (rt / k)     # actualizar coeficiente: c_{k} = c_{k-1} * (rt)/k
        B = B + c * A        # agregar término k
        A = A @ hatP         # siguiente potencia: hatP^{k+1}
        suma = suma + c
        k += 1
    M = k - 1   # porque k se incrementa después de usar el término
    return B, M

# Calcular para t = 0.5, 1, 5
t_values = [0.5, 1, 5]
epsilon = 1e-5
for t in t_values:
    Pt, M = P_t_epsilon(t, r, hatP, eps=epsilon)
    print(f"\n--- t = {t} ---")
    print(f"M requerido = {M} (términos 0..{M})")
    print("P(t) aproximada:")
    print(np.round(Pt, 6))


--- t = 0.5 ---
M requerido = 13 (términos 0..13)
P(t) aproximada:
[[0.2506 0.217  0.3867 0.1458]
 [0.2531 0.2384 0.3744 0.1341]
 [0.1691 0.1936 0.4203 0.217 ]
 [0.158  0.1574 0.3983 0.2862]]

--- t = 1 ---
M requerido = 19 (términos 0..19)
P(t) aproximada:
[[0.2061 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]

--- t = 5 ---
M requerido = 56 (términos 0..56)
P(t) aproximada:
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


Explicación del algoritmo:



*   El bucle itera hasta que la suma de los coeficientes de Poisson (probabilidad de que $N(t)\leq M)$ sea al menos $1-\epsilon$. Esto garantiza que la cola.
$\sum^\infty_{k=M+1} e^{-rt} *\frac{(rt)^k}{k!} \leq \epsilon$
*   Cada iteración calcula el siguiente término de la serie y lo acumula en
$B$
*  Al final, $M$ es el número de términos utilizados (desde $k=0$ hasta $k=M$).